In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_large/articles.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_large/validation/history.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_large/validation/behaviors.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_large/train/history.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_large/train/behaviors.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_testset/ebnerd_testset/articles.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_testset/ebnerd_testset/test/history.parquet
/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_testset/ebnerd_testset/test/behaviors.parquet
/kaggle/input/datasets/wrathofgod123/trained-model/ebnerd_large_lgbm.txt
/kaggle/input/datasets/wrathofgod123/trained-model/__huggingface_repos__.json


In [2]:
!pip install lightgbm sentence-transformers -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from collections import Counter
from sentence_transformers import SentenceTransformer

TEST ="/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_testset/ebnerd_testset"
MODEL="/kaggle/input/datasets/wrathofgod123/trained-model/ebnerd_large_lgbm.txt"
def pfx(x): return f"eb:{x}"

# ---- load trained model ----
rk = lgb.Booster(model_file=MODEL)
print("model loaded:", MODEL, "| features:", rk.num_feature())

# ---- testset articles (fresh test-period stats) ----
at=pl.read_parquet(f"{TEST}/articles.parquet")
pub={pfx(r):p for r,p in zip(at["article_id"].to_list(),at["published_time"].to_list())}
pv ={pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(),at["total_pageviews"].to_list())}
iv ={pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(),at["total_inviews"].to_list())}
rt ={pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(),at["total_read_time"].to_list())}
cat={pfx(r):(c or "") for r,c in zip(at["article_id"].to_list(),at["category_str"].to_list())}
sent={pfx(r):(s or 0.0) for r,s in zip(at["article_id"].to_list(),at["sentiment_score"].to_list())}
txt={pfx(r):f"{t or ''} {s or ''}".strip() for r,t,s in zip(at["article_id"].to_list(),at["title"].to_list(),at["subtitle"].to_list())}
pv_logmax=np.log1p(max([v for v in pv.values() if v>0] or [1]))
print("testset articles:",len(pub))

# ---- MiniLM multilingual (same as training) ----
minilm=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
aids=list(txt.keys()); atxt=[txt[i] if txt[i] else "nyhed" for i in aids]
emb=minilm.encode(atxt,batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb_by_id={aids[i]:emb[i] for i in range(len(aids))}
print("MiniLM encoded")

def recency(aid,T,tau=24.0):
    p=pub.get(aid)
    if p is None or T is None: return 0.0
    dh=(T-p).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def load_hist(base):
    h=pl.read_parquet(f"{base}/history.parquet")
    return {pfx(u):[pfx(x) for x in (arts or [])] for u,arts in
            zip(h["user_id"].to_list(), h["article_id_fixed"].to_list())}
def user_prof(hist,mh=30):
    ai=hist[-mh:] if hist else []
    cats=[cat.get(x) for x in ai];tot=len([c for c in cats if c])
    cc=Counter(c for c in cats if c);cd={k:v/tot for k,v in cc.items()} if tot else {}
    hv=[emb_by_id[x] for x in ai if x in emb_by_id]
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    return cd,um,hv
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

def build_feats(uid,T,cand,hl):
    cd,um,hv=user_prof(hl.get(uid,[]));m=len(cand);F=[]
    for i,c in enumerate(cand):
        rec=recency(c,T)
        p=np.log1p(pv.get(c,0))/pv_logmax
        ivn=np.log1p(iv.get(c,0))/pv_logmax
        rtn=np.log1p(rt.get(c,0))/pv_logmax
        ctr=pv.get(c,0)/iv.get(c,1) if iv.get(c,0)>0 else 0.0
        cm=cd.get(cat.get(c,""),0.0)
        cv=emb_by_id.get(c)
        em=float(um@cv) if (um is not None and cv is not None) else 0.0
        eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
        sn=sent.get(c,0.0);pos=i/max(1,m-1)
        F.append([rec,p,ivn,rtn,ctr,cm,em,eb,sn,pos,m])
    F=np.array(F); F[:,0]=mm(F[:,0])
    return F

# ---- predict on 13.5M testset ----
hist_test=load_hist(f"{TEST}/test")
b_te=pl.read_parquet(f"{TEST}/test/behaviors.parquet",
                     columns=["impression_id","user_id","impression_time","article_ids_inview"])
print("predicting",b_te.height,"test impressions...")
ci=0
with open("/kaggle/working/predictions.txt","w") as fout:
    for iid,uid,T,inv in zip(b_te["impression_id"].to_list(),b_te["user_id"].to_list(),
                             b_te["impression_time"].to_list(),b_te["article_ids_inview"].to_list()):
        cand=[pfx(x) for x in inv]
        sc=rk.predict(build_feats(pfx(uid),T,cand,hist_test))
        order=np.argsort(-sc);ranks=np.empty(len(cand),int);ranks[order]=np.arange(1,len(cand)+1)
        fout.write(f"{iid} [{','.join(map(str,ranks.tolist()))}]\n")
        ci+=1
        if ci%1000000==0: print(f"  {ci}/{b_te.height}")
with zipfile.ZipFile("/kaggle/working/ebnerd_large_submission.zip","w",zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/predictions.txt","predictions.txt")
print(f"DONE. {ci} -> ebnerd_large_submission.zip")

# ---- HF backup ----
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi, login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    api=HfApi(); api.create_repo("donbosoc/ebnerd-artifacts",repo_type="dataset",exist_ok=True,private=True)
    api.upload_file(path_or_fileobj="/kaggle/working/ebnerd_large_submission.zip",
                    path_in_repo="ebnerd_large_submission.zip",
                    repo_id="donbosoc/ebnerd-artifacts",repo_type="dataset")
    print("HF backup done")
except Exception as e:
    print("HF push failed (commit persists):", e)

model loaded: /kaggle/input/datasets/wrathofgod123/trained-model/ebnerd_large_lgbm.txt | features: 11
testset articles: 125541


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/246 [00:00<?, ?it/s]

MiniLM encoded
predicting 13536710 test impressions...
  1000000/13536710
  2000000/13536710
  3000000/13536710
  4000000/13536710
  5000000/13536710
  6000000/13536710
  7000000/13536710
  8000000/13536710
  9000000/13536710
  10000000/13536710
  11000000/13536710
  12000000/13536710
  13000000/13536710
DONE. 13536710 -> ebnerd_large_submission.zip
HF push failed (commit persists): Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 131628449 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.
